# cAPTure: Stage-2 training preflight v2

This notebook freezes and audits bounded chronological validation blocks before any model training. For each development scenario, it selects the latest admissible 20% block, falling back to 25% only when necessary. It refuses to split attack-step iterations, excludes the later suffix during epoch selection, verifies graph/packet alignment, and runs inference-only synthetic model-interface checks.

It does not optimize model parameters, select score thresholds, or access Test1, Test2, `train_pub_exf`, or `train_user_prop`.

## 1. Mount Drive and prepare the environment

Push the current branch before opening Colab. Use a fresh runtime so imported modules cannot come from an older commit.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_GRAPH_PARENT = Path("/content/capture_graph_stage2_inputs_v2")
LOCAL_PREFLIGHT_WORK_ROOT = Path("/content/capture_graph_preflight_work")

repository_already_present = PROJECT_ROOT.exists()
if not repository_already_present:
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )
if repository_already_present:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
        cwd=PROJECT_ROOT,
        check=True,
    )
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")

required_files = [
    "code/python/requirements-capture-mlp.txt",
    "code/python/utils/capture_graph_dataset.py",
    "code/python/utils/capture_graph_training_preflight.py",
    "code/python/utils/capture_graph_training_preflight_v2.py",
    "configs/capture_graph_stage2_input_v2.yaml",
    "configs/capture_graph_training_preflight_v2.yaml",
]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture-mlp.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
from IPython.display import display
from utils.capture_graph_dataset import stage_capture_graph_materialization
from utils.capture_graph_training_preflight_v2 import (
    load_completed_training_preflight_v2,
    run_capture_graph_training_preflight_v2,
    save_training_preflight_v2_decision,
)

print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("Training-preflight environment is ready.")

Mounted at /content/drive
Repository commit: b403aed0f571941364ff6c8420d6cddeb6d9ac87
Training-preflight environment is ready.


## 2. Bind the reviewed immutable inputs

The preflight contract names the exact prepared, graph-materialization, and graph-input-audit runs. A different run ID or checksum is rejected.

In [2]:
PREPARED_RUN_ID = "20260917T235058_827743Z_prepare_full_dev"
GRAPH_MATERIALIZATION_RUN_ID = (
    "20260928T175821_521784Z_graph_materialization_full_dev"
)
GRAPH_INPUT_AUDIT_RUN_ID = "20260928T193308_017682Z_graph_input_full"

PREPARED_RUN_DIR = DRIVE_ROOT / "prepared_runs" / PREPARED_RUN_ID
GRAPH_MATERIALIZATION_DIR = (
    DRIVE_ROOT / "graph_materialization_runs" / GRAPH_MATERIALIZATION_RUN_ID
)
GRAPH_INPUT_AUDIT_DIR = (
    DRIVE_ROOT / "graph_input_audit_runs" / GRAPH_INPUT_AUDIT_RUN_ID
)
GRAPH_INPUT_CONTRACT_PATH = (
    PROJECT_ROOT / "configs/capture_graph_stage2_input_v2.yaml"
)
PREFLIGHT_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_training_preflight_v2.yaml"
)
for path in (
    PREPARED_RUN_DIR, GRAPH_MATERIALIZATION_DIR, GRAPH_INPUT_AUDIT_DIR,
    GRAPH_INPUT_CONTRACT_PATH, PREFLIGHT_CONFIG_PATH,
):
    if not path.exists():
        raise FileNotFoundError(f"Required preflight input is missing: {path}")

print("Prepared packets:", PREPARED_RUN_DIR)
print("Graph materialization:", GRAPH_MATERIALIZATION_DIR)
print("Graph-input audit:", GRAPH_INPUT_AUDIT_DIR)
print("Preflight contract:", PREFLIGHT_CONFIG_PATH)

Prepared packets: /content/drive/MyDrive/capture_gate0/prepared_runs/20260917T235058_827743Z_prepare_full_dev
Graph materialization: /content/drive/MyDrive/capture_gate0/graph_materialization_runs/20260928T175821_521784Z_graph_materialization_full_dev
Graph-input audit: /content/drive/MyDrive/capture_gate0/graph_input_audit_runs/20260928T193308_017682Z_graph_input_full
Preflight contract: /content/temporalgnn-nids/configs/capture_graph_training_preflight_v2.yaml


## 3. Stage and verify the graph collection

The existing FULL_DEV materialization is copied to local Colab storage and every artifact checksum is verified. No graph is rebuilt.

In [3]:
collection = stage_capture_graph_materialization(
    source_root=GRAPH_MATERIALIZATION_DIR,
    local_parent=LOCAL_GRAPH_PARENT,
    contract_path=GRAPH_INPUT_CONTRACT_PATH,
    expected_run_id=GRAPH_MATERIALIZATION_RUN_ID,
)
print("Verified local graph root:", collection.root)
print("All graph checksums verified:",
      collection.artifact_checksums_verified)

Staged and verified graph materialization: /content/capture_graph_stage2_inputs_v2/20260928T175821_521784Z_graph_materialization_full_dev
Verified local graph root: /content/capture_graph_stage2_inputs_v2/20260928T175821_521784Z_graph_materialization_full_dev
All graph checksums verified: True


## 4. Run or resume the training preflight

This scans only prepared development metadata, finds the latest admissible bounded validation block for each scenario, verifies graph positions for the training, validation, and excluded suffix partitions, and performs inference-only synthetic forward/reset checks. It may take several minutes because the prepared artifacts are checksum-verified and staged one scenario at a time.

In [4]:
PREFLIGHT_RUN_ID = None  # Paste an existing completed v2 ID to reuse it.
if PREFLIGHT_RUN_ID is None:
    PREFLIGHT_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_graph_training_preflight_v2"
    )
PREFLIGHT_OUTPUT_DIR = (
    DRIVE_ROOT / "graph_training_preflight_v2_runs" / PREFLIGHT_RUN_ID
)
print("Training-preflight run ID:", PREFLIGHT_RUN_ID)
print("Training-preflight output:", PREFLIGHT_OUTPUT_DIR)

if PREFLIGHT_OUTPUT_DIR.exists():
    preflight_report = load_completed_training_preflight_v2(
        PREFLIGHT_OUTPUT_DIR,
        collection=collection,
        config_path=PREFLIGHT_CONFIG_PATH,
    )
    print("Reused the completed immutable training preflight.")
else:
    preflight_report = run_capture_graph_training_preflight_v2(
        collection=collection,
        prepared_run_dir=PREPARED_RUN_DIR,
        graph_input_audit_dir=GRAPH_INPUT_AUDIT_DIR,
        config_path=PREFLIGHT_CONFIG_PATH,
        output_dir=PREFLIGHT_OUTPUT_DIR,
        local_work_root=LOCAL_PREFLIGHT_WORK_ROOT,
    )
    print("Saved the immutable training preflight.")

Training-preflight run ID: 20260929T131750_353987Z_graph_training_preflight_v2
Training-preflight output: /content/drive/MyDrive/capture_gate0/graph_training_preflight_v2_runs/20260929T131750_353987Z_graph_training_preflight_v2
Profiled validation blocks for train_empty_conn.
Profiled validation blocks for train_qos_mid.
Profiled validation blocks for train_dollar_char.
Profiled validation blocks for train_slash_char.
Profiled validation blocks for train_sub_exf.
Saved the immutable training preflight.


## 5. Review candidate blocks and selected boundaries

Each scenario independently selects the smallest declared block fraction that has an admissible block, then uses its latest admissible position. Verify both packet labels, benign-only and attack-containing windows, complete iterations, multiple steps, and the size of the suffix excluded during epoch selection.

In [5]:
candidate_table = pd.read_csv(PREFLIGHT_OUTPUT_DIR / "split_candidates.csv")
try:
    split_table = pd.read_csv(PREFLIGHT_OUTPUT_DIR / "split_summary.csv")
except pd.errors.EmptyDataError:
    # Older blocked preflight runs wrote this artifact without a header.
    split_table = pd.DataFrame()
display(candidate_table)
print("Preflight status:", preflight_report["status"])
print("Selected validation block fraction by scenario:", json.dumps(
    preflight_report["selected_validation_block_fraction_by_scenario"],
    indent=2,
))
print("Fold totals:", json.dumps(preflight_report["fold_totals"], indent=2))
print("Checkpointing contract:", json.dumps(
    preflight_report["checkpointing"], indent=2
))
print("Final-refit contract:", json.dumps(
    preflight_report["final_refit"], indent=2
))
if split_table.empty:
    print("No complete set of scenario blocks was selected. Review the rejection reasons above.")
else:
    display(split_table)

,scenario,candidate_validation_block_fraction,target_validation_block_windows,validation_start_window_index,validation_end_window_index_exclusive,realized_validation_block_fraction,post_validation_wall_clock_windows,admissible,rejection_reasons,split_iterations,...,inner_validation.distinct_attack_steps,post_validation.wall_clock_windows,post_validation.nonempty_windows,post_validation.benign_only_windows,post_validation.attack_containing_windows,post_validation.packets,post_validation.normal_packets,post_validation.attack_packets,post_validation.complete_attack_iterations,post_validation.distinct_attack_steps
0,train_empty_conn,0.20,1872,2529,4401,0.20,4959,True,[],[],...,4,4959,4959,2473,2486,413336,390347,22989,8,3
1,train_empty_conn,0.25,2340,3063,5403,0.25,3957,True,[],[],...,4,3957,3957,2416,1541,322196,307274,14922,6,3
2,train_qos_mid,0.20,1872,6716,8588,0.20,772,True,[],[],...,3,772,772,772,0,56884,56884,0,0,0
3,train_qos_mid,0.25,2340,6716,9056,0.25,304,True,[],[],...,3,304,304,304,0,21849,21849,0,0,0
4,train_dollar_char,0.20,1872,7452,9324,0.20,36,True,[],[],...,3,36,33,33,0,151,151,0,0,0
5,train_dollar_char,0.25,2340,7019,9359,0.25,1,True,[],[],...,3,1,1,1,0,2,2,0,0,0
6,train_slash_char,0.20,1872,7480,9352,0.20,8,True,[],[],...,2,8,7,7,0,14,14,0,0,0
7,train_slash_char,0.25,2340,7020,9360,0.25,0,True,[],[],...,2,0,0,0,0,0,0,0,0,0
8,train_sub_exf,0.20,1872,7461,9333,0.20,27,True,[],[],...,4,27,25,25,0,106,106,0,0,0
9,train_sub_exf,0.25,2340,7012,9352,0.25,8,True,[],[],...,4,8,7,7,0,14,14,0,0,0


Preflight status: review_required
Selected validation block fraction by scenario: {
  "train_empty_conn": 0.2,
  "train_qos_mid": 0.2,
  "train_dollar_char": 0.2,
  "train_slash_char": 0.2,
  "train_sub_exf": 0.2
}
Fold totals: {
  "A": {
    "selection_inner_train_edges": 1820681,
    "selection_inner_validation_edges": 384595,
    "selection_excluded_post_validation_edges": 470220,
    "complete_refit_edges": 2675496,
    "complete_refit_graphs": 18720
  },
  "B": {
    "selection_inner_train_edges": 10974255,
    "selection_inner_validation_edges": 559352,
    "selection_excluded_post_validation_edges": 271,
    "complete_refit_edges": 11533878,
    "complete_refit_graphs": 27347
  }
}
Checkpointing contract: {
  "selection_unit": "model_and_fold",
  "metric": "unweighted_mean_scenario_average_precision",
  "maximum_epochs": 60,
  "minimum_epochs": 10,
  "patience_epochs": 10,
  "minimum_absolute_improvement": 0.0001,
  "restore_best_epoch": true,
  "preprocessing_fit_scope": "inner

,scenario,fold,validation_start_window_index,validation_end_window_index_exclusive,selected_validation_block_fraction,realized_validation_block_fraction,inner_train_graphs,inner_validation_graphs,post_validation_graphs,inner_train_edges,...,inner_validation.distinct_attack_steps,post_validation.wall_clock_windows,post_validation.nonempty_windows,post_validation.benign_only_windows,post_validation.attack_containing_windows,post_validation.packets,post_validation.normal_packets,post_validation.attack_packets,post_validation.complete_attack_iterations,post_validation.distinct_attack_steps
0,train_empty_conn,A,2529,4401,0.2,0.2,2529,1872,4959,593753,...,4,4959,4959,2473,2486,413336,390347,22989,8,3
1,train_qos_mid,A,6716,8588,0.2,0.2,6716,1872,772,1226928,...,3,772,772,772,0,56884,56884,0,0,0
2,train_dollar_char,B,7452,9324,0.2,0.2,7451,1632,33,2759427,...,3,36,33,33,0,151,151,0,0,0
3,train_slash_char,B,7480,9352,0.2,0.2,7478,1631,7,6074640,...,2,8,7,7,0,14,14,0,0,0
4,train_sub_exf,B,7461,9333,0.2,0.2,7459,1631,25,2140188,...,4,27,25,25,0,106,106,0,0,0


## 6. Review the synthetic model-interface contract

Every variant must produce one finite logit per synthetic edge without mutating shared inputs. Temporal variants must create and then completely reset their state. No backward pass or optimizer is used.

In [6]:
model_contract = preflight_report["model_contract"]
model_rows = [
    {"variant": name, **result}
    for name, result in model_contract["variants"].items()
]
display(pd.DataFrame(model_rows))
print("Shared inputs unchanged:", model_contract["shared_inputs_unchanged"])
print("Optimization performed:", model_contract["optimization_performed"])
print("Held-out scenarios accessed:",
      preflight_report["held_out_scenarios_accessed"])
print("Model training performed:",
      preflight_report["model_training_performed"])
print("Selection features materialized:",
      preflight_report["selection_features_materialized"])
print("Final refit performed:",
      preflight_report["final_refit_performed"])

,variant,status,temporal,temporal_state_created,temporal_reset_verified,logits,parameters
0,edge_mlp,passed,False,False,False,4,4545
1,edge_gru,passed,True,True,True,4,16257
2,static_gnn,passed,False,False,False,4,24097
3,st_gnn,passed,True,True,True,4,30433
4,st_gnn_without_gat,passed,True,True,True,4,13921
5,st_gnn_without_direct_edge_attr,passed,True,True,True,4,27137


Shared inputs unchanged: True
Optimization performed: False
Held-out scenarios accessed: False
Model training performed: False
Selection features materialized: False
Final refit performed: False


## 7. Record the manual preflight decision

Approval freezes the scenario blocks, the per-model/per-fold best-epoch rule, and the complete-data final-refit rule. It authorizes creation of the full training runner but does not authorize training yet. Do not approve a blocked preflight.

In [7]:
APPROVE_TRAINING_PREFLIGHT = True
PREFLIGHT_REVIEW_NOTE = (
    "Approved for pilot implementation. Every development scenario has an "
    "iteration-safe 20% chronological validation block with the required "
    "class, window, iteration, and step support. Graph partitions conserve "
    "all fold-training edges. The large post-validation suffix in "
    "train_empty_conn is accepted because it remains excluded during epoch "
    "selection and is restored only during the frozen complete-data refit. "
    "No model training, model scores, outer-validation scenarios, or held-out "
    "scenarios were used."
)
DECISION_PATH = PREFLIGHT_OUTPUT_DIR / "training_preflight_decision.json"

if DECISION_PATH.exists():
    decision = json.loads(DECISION_PATH.read_text(encoding="utf-8"))
    print("Existing preflight decision:", json.dumps(decision, indent=2))
else:
    if not APPROVE_TRAINING_PREFLIGHT:
        raise RuntimeError(
            "Review the preflight, then explicitly approve or retain the run as unapproved."
        )
    decision = save_training_preflight_v2_decision(
        PREFLIGHT_OUTPUT_DIR,
        approved=True,
        review_note=PREFLIGHT_REVIEW_NOTE,
    )
    print("Saved preflight decision:", json.dumps(decision, indent=2))

Saved preflight decision: {
  "approved": true,
  "review_note": "Approved for pilot implementation. Every development scenario has an iteration-safe 20% chronological validation block with the required class, window, iteration, and step support. Graph partitions conserve all fold-training edges. The large post-validation suffix in train_empty_conn is accepted because it remains excluded during epoch selection and is restored only during the frozen complete-data refit. No model training, model scores, outer-validation scenarios, or held-out scenarios were used.",
  "training_preflight_report_sha256": "478b6c4cae0ee27595c2ddd7fea73e2c0c3914f14c78f60ac4534af77badafed",
  "selected_validation_block_fraction_by_scenario": {
    "train_empty_conn": 0.2,
    "train_qos_mid": 0.2,
    "train_dollar_char": 0.2,
    "train_slash_char": 0.2,
    "train_sub_exf": 0.2
  },
  "checkpointing": {
    "selection_unit": "model_and_fold",
    "metric": "unweighted_mean_scenario_average_precision",
    "